In [1]:
import cv2
import torch
import logging
import os
from datetime import datetime
from ultralytics import YOLO, solutions

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def process_video(model_path, video_path, save_dir="../output"):
    """视频处理生成器，返回视频参数和帧数据"""
    # 创建视频专属目录
    video_base = os.path.basename(video_path)
    video_name = os.path.splitext(video_base)[0]
    main_save_path = os.path.join(save_dir, video_name)
    os.makedirs(main_save_path, exist_ok=True)

    model = YOLO(model_path)
    model.to(device)

    capture = cv2.VideoCapture(video_path)
    if not capture.isOpened():
        logging.error("Error 打开视频错误")
        return

    # 视频参数获取
    fps = capture.get(cv2.CAP_PROP_FPS)
    w = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    line_points = [(0, h//2), (w, h//2)]

    # 初始化ObjectCounter
    counter = solutions.ObjectCounter(
        view_img=False,
        reg_pts=line_points,
        names=model.names,
        draw_tracks=True,
        line_thickness=1,
    )

    # 跟踪参数
    seen_ids = set()
    frame_count = 0  # 手动维护帧计数器

    yield (fps, w, h)

    # 视频处理循环
    while capture.isOpened():
        success, frame = capture.read()
        frame_count += 1  # 帧号从1开始计数
        if not success:
            logging.warning("视频帧读取结束")
            break

        # 目标跟踪
        tracks = model.track(frame, persist=True, show=False, device=device, classes=[2,3,5,7])
        # processed_frame = counter.start_counting(frame, tracks)

        # 首次出现物体处理
        if tracks and tracks[0].boxes is not None:
            for box in tracks[0].boxes:
                obj_id = int(box.id.item())
                if obj_id not in seen_ids:
                    # 创建对象专属目录
                    cls_id = int(box.cls.item())
                    class_name = model.names[cls_id]
                    obj_dir = os.path.join(
                        main_save_path,
                        f"ID{obj_id}_{class_name}_{datetime.now().strftime('%Y%m%d%H%M%S')}" #时间戳
                    )
                    os.makedirs(obj_dir, exist_ok=True)

                    # 保存物体裁剪
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    crop = frame[y1:y2, x1:x2]
                    cv2.imwrite(os.path.join(obj_dir, "ID{obj_id}_{class_name}.jpg"), crop)

                    # 保存完整帧
                    frame_filename = f"frame_{frame_count:06d}.jpg" # 第frame_count帧
                    cv2.imwrite(os.path.join(obj_dir, frame_filename), frame)

                    seen_ids.add(obj_id)
                
        processed_frame = counter.start_counting(frame, tracks)  # 传入原始frame进行绘制
        
        counts = {
            "in": counter.in_counts,
            "out": counter.out_counts,
            "class_wise": counter.class_wise_count.copy()
        }
        yield (processed_frame, counts)

    capture.release()

def display_and_print(generator):
    """显示画面并打印计数信息"""
    try:
        # 获取视频基础参数
        fps, w, h = next(generator)
    except StopIteration:
        return

    while True:
        try:
            frame, counts = next(generator)
        except StopIteration:
            break

        # 显示画面
        cv2.imshow('output', frame)
        
        # 打印信息
        print(f"\n当前统计:")
        print(f"总进入(下到上): {counts['in']} | 总离开(上到下): {counts['out']}")
        for cls_name, cls_counts in counts['class_wise'].items():
            print(f"{cls_name} 进入: {cls_counts.get('IN',0)} | 离开: {cls_counts.get('OUT',0)}")

        # 退出控制，按q键
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cv2.destroyAllWindows()

In [2]:
# 初始化生成器（指定保存目录）
gen = process_video(
    model_path="yolo11n.pt",
    video_path="../kuruma.mp4",
    save_dir="../output"  # 截图保存目录
)

# 正常处理流程
display_and_print(gen)

Line Counter Initiated.

0: 384x640 9 cars, 2 buss, 35.4ms
Speed: 4.0ms preprocess, 35.4ms inference, 155.3ms postprocess per image at shape (1, 3, 384, 640)

当前统计:
总进入(下到上): 0 | 总离开(上到下): 0
bus 进入: 0 | 离开: 0
car 进入: 0 | 离开: 0

0: 384x640 9 cars, 2 buss, 13.0ms
Speed: 3.5ms preprocess, 13.0ms inference, 4.5ms postprocess per image at shape (1, 3, 384, 640)

当前统计:
总进入(下到上): 0 | 总离开(上到下): 2
bus 进入: 0 | 离开: 1
car 进入: 0 | 离开: 1

0: 384x640 9 cars, 2 buss, 14.0ms
Speed: 2.0ms preprocess, 14.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

当前统计:
总进入(下到上): 0 | 总离开(上到下): 2
bus 进入: 0 | 离开: 1
car 进入: 0 | 离开: 1

0: 384x640 9 cars, 2 buss, 12.0ms
Speed: 2.2ms preprocess, 12.0ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

当前统计:
总进入(下到上): 0 | 总离开(上到下): 2
bus 进入: 0 | 离开: 1
car 进入: 0 | 离开: 1

0: 384x640 9 cars, 2 buss, 11.8ms
Speed: 3.5ms preprocess, 11.8ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

当前统计:
总进入(下到上): 0 | 总离开(上到下): 2
bus 进


当前统计:
总进入(下到上): 15 | 总离开(上到下): 23
bus 进入: 1 | 离开: 1
car 进入: 14 | 离开: 21
truck 进入: 0 | 离开: 1
